# Aufgabe 3 – SciQ Scorecards Chatbot

In [1]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
torch.manual_seed(42)

Device: cpu


## 1 – Datensatz laden

In [2]:
dataset = load_dataset("allenai/sciq")
ex = dataset["train"][0]
print("Frage:",    ex["question"])
print("Korrekt:",  ex["correct_answer"])
print("Support:",  ex["support"][:80], "...")
print("Splits:",   {k: len(v) for k, v in dataset.items()})

Frage: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Korrekt: mesophilic organisms
Support: Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (7 ...
Splits: {'train': 11679, 'validation': 1000, 'test': 1000}


## 2 – Tokenizer & Embedding-Matrix

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
hf_model  = AutoModel.from_pretrained("bert-base-uncased")

emb_matrix = hf_model.get_input_embeddings().weight.detach().cpu()
EMB_DIM    = emb_matrix.shape[1]   # 768
del hf_model

MAX_Q, MAX_S, MAX_A = 48, 128, 28
print(f"Embedding-Matrix: {emb_matrix.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding-Matrix: torch.Size([30522, 768])


## 3 – Hilfsfunktionen

In [4]:
def tokenize_text(text, max_len):
    if not text:
        return []
    return tokenizer.encode(text, add_special_tokens=False,
                            truncation=True, max_length=max_len)

def mean_embedding(token_ids):
    if not token_ids:
        return torch.zeros(EMB_DIM)
    ids = torch.tensor(token_ids)
    return emb_matrix[ids].mean(0)

def build_feature(ctx, ans):
    """[ctx | ans | |ctx-ans| | ctx*ans]  →  (3072,)"""
    return torch.cat([ctx, ans, (ctx - ans).abs(), ctx * ans])

v = build_feature(mean_embedding(tokenize_text("test", MAX_Q)),
                  mean_embedding(tokenize_text("answer", MAX_A)))
print("Feature-Vektor Shape:", v.shape)  # (3072,)

Feature-Vektor Shape: torch.Size([3072])


## 4 – Dataset & Collate

In [5]:
class SciQDataset(Dataset):
    def __init__(self, split_data, split):
        self.data  = split_data
        self.train = (split == "train")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        answers = [ex["correct_answer"],
                   ex["distractor1"],
                   ex["distractor2"],
                   ex["distractor3"]]

        if self.train:
            perm = torch.randperm(4).tolist()
        else:
            rng = random.Random(42 + idx)
            perm = list(range(4))
            rng.shuffle(perm)

        permuted = [answers[i] for i in perm]
        label    = perm.index(0)         

        return {
            "q_ids":  tokenize_text(ex["question"],         MAX_Q),
            "s_ids":  tokenize_text(ex.get("support", ""), MAX_S),
            "a_ids":  [tokenize_text(a, MAX_A) for a in permuted],
            "label":  label,
        }


def collate(batch):
    features, labels = [], []
    for ex in batch:
        ctx  = mean_embedding(ex["q_ids"] + ex["s_ids"])          
        feat = torch.stack(
            [build_feature(ctx, mean_embedding(a)) for a in ex["a_ids"]]
        )                                                          
        features.append(feat)
        labels.append(ex["label"])
    return torch.stack(features), torch.tensor(labels)             

In [6]:
BATCH = 16

train_ds = SciQDataset(dataset["train"],      "train")
val_ds   = SciQDataset(dataset["validation"], "val")
test_ds  = SciQDataset(dataset["test"],       "test")

train_loader = DataLoader(train_ds, batch_size=BATCH,  shuffle=True,  collate_fn=collate, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH*2, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH*2, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)

X, y = next(iter(train_loader))
print("X:", X.shape, "| y:", y.shape)  

X: torch.Size([16, 4, 3072]) | y: torch.Size([16])


/Users/lucius.lechner/GitHub/lernverfahren/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## 5 – Modell

In [7]:
F_DIM = EMB_DIM * 4  

class ScorecardsMLP(nn.Module):
    def __init__(self, f=F_DIM, h1=1024, h2=512, drop=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(f,  h1), nn.LayerNorm(h1), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(h1, h2), nn.LayerNorm(h2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(h2,  1),
        )

    def forward(self, x):          
        B = x.shape[0]
        return self.net(x.view(B * 4, -1)).view(B, 4)  

model = ScorecardsMLP().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameter: {total_params:,}")
print("Logits-Shape:", model(X.to(DEVICE)).shape)   

Parameter: 3,675,137
Logits-Shape: torch.Size([16, 4])


## 6 – Training

In [8]:
EPOCHS = 50
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)


def evaluate(mdl, loader):
    mdl.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            correct += (mdl(X).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


best_val, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    val_acc = evaluate(model, val_loader)
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch:3d}/{EPOCHS} | Loss {avg_loss:.4f} | Val {val_acc:.4f} | Best-Val {best_val:.4f}")

print(f"\nBeste Validation Accuracy: {best_val:.4f}")

Epoch   1/50 | Loss 1.3593 | Val 0.4190 | Best-Val 0.4190
Epoch   2/50 | Loss 1.2923 | Val 0.4510 | Best-Val 0.4510
Epoch   3/50 | Loss 1.2623 | Val 0.4930 | Best-Val 0.4930
Epoch   4/50 | Loss 1.2304 | Val 0.5070 | Best-Val 0.5070
Epoch   5/50 | Loss 1.2055 | Val 0.5170 | Best-Val 0.5170
Epoch   6/50 | Loss 1.1791 | Val 0.5160 | Best-Val 0.5170
Epoch   7/50 | Loss 1.1484 | Val 0.5550 | Best-Val 0.5550
Epoch   8/50 | Loss 1.1233 | Val 0.5600 | Best-Val 0.5600
Epoch   9/50 | Loss 1.0929 | Val 0.5760 | Best-Val 0.5760
Epoch  10/50 | Loss 1.0641 | Val 0.5930 | Best-Val 0.5930
Epoch  11/50 | Loss 1.0344 | Val 0.5960 | Best-Val 0.5960
Epoch  12/50 | Loss 1.0061 | Val 0.6170 | Best-Val 0.6170
Epoch  13/50 | Loss 0.9766 | Val 0.6030 | Best-Val 0.6170
Epoch  14/50 | Loss 0.9481 | Val 0.6180 | Best-Val 0.6180
Epoch  15/50 | Loss 0.9148 | Val 0.6300 | Best-Val 0.6300
Epoch  16/50 | Loss 0.8853 | Val 0.6580 | Best-Val 0.6580
Epoch  17/50 | Loss 0.8626 | Val 0.6480 | Best-Val 0.6580
Epoch  18/50 |

## 7 – Test-Evaluation

In [9]:
model.load_state_dict(best_state)

train_acc = evaluate(model, train_loader)
val_acc   = evaluate(model, val_loader)
test_acc  = evaluate(model, test_loader)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Val   Accuracy : {val_acc:.4f}")
print(f"Test  Accuracy : {test_acc:.4f}")

Train Accuracy : 0.9877
Val   Accuracy : 0.7410
Test  Accuracy : 0.7330


## 8 – ask_bot & drei automatische Demo-Aufrufe

In [23]:
def ask_bot(question, options, support=""):
    assert len(options) == 4, "Genau 4 Optionen erforderlich"
    model.eval()
    ctx  = mean_embedding(tokenize_text(question, MAX_Q) + tokenize_text(support, MAX_S))
    feat = torch.stack(
        [build_feature(ctx, mean_embedding(tokenize_text(o, MAX_A))) for o in options]
    ).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(feat), dim=1)[0]
    best = probs.argmax().item()
    print(f"  → Antwort: {options[best]}")
    for i, (o, p) in enumerate(zip(options, probs)):
        marker = "✓" if i == best else " "
        print(f"    [{marker}] {p:.3f}  {o}")
    return options[best]


def run_three_ask_bot_examples():
    print("=" * 60)
    indices = random.sample(range(len(dataset["test"])), 3)  
    for i, idx in enumerate(indices):
        ex = dataset["test"][idx]
        options = [ex["correct_answer"], ex["distractor1"], ex["distractor2"], ex["distractor3"]]
        random.shuffle(options)  
        print(f"\nBeispiel {i+1}: {ex['question']}")
        print(f"  Erwartet : {ex['correct_answer']}")
        ask_bot(ex["question"], options, ex.get("support", ""))
        print("=" * 60)


run_three_ask_bot_examples()


Beispiel 1: What must be combined with a halogen to give it a positive oxidation number?
  Erwartet : oxygen
  → Antwort: oxygen
    [ ] 0.145  carbon
    [ ] 0.163  calcium
    [✓] 0.519  oxygen
    [ ] 0.174  nitrogen

Beispiel 2: How does the water cycle end?
  Erwartet : it repeats itself
  → Antwort: ocean evaporation
    [ ] 0.121  cloud precipitation
    [✓] 0.685  ocean evaporation
    [ ] 0.131  freezing glaciers
    [ ] 0.063  it repeats itself

Beispiel 3: What is a body of freshwater that flows downhill in a channel?
  Erwartet : a stream
  → Antwort: a stream
    [ ] 0.155  a river
    [ ] 0.335  a creek
    [ ] 0.062  an eddy
    [✓] 0.448  a stream


In [13]:
torch.save(best_state, "../models/task3/sciq_mlp.pt")

In [15]:
model = ScorecardsMLP().to(DEVICE)

model.load_state_dict(torch.load("../models/task3/sciq_mlp.pt", map_location=DEVICE))

test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.7330
